<a href="https://colab.research.google.com/github/shehabmariam87-lab/Titanic-Survival-Prediction/blob/main/Titanic_Survivals_by_Mariam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# TITANIC - XGBOOST COMPLETE CODE
# Cleaning → Training → Accuracy → Submission
# ============================================

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# --------------------------------------------
# 1. LOAD DATA
# --------------------------------------------

train = pd.read_csv(r"C:\Users\sheha\Titanic\train.csv")
test = pd.read_csv(r"C:\Users\sheha\Titanic\test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# --------------------------------------------
# 2. CLEANING
# --------------------------------------------

# Fill missing Age using training median
train["Age"] = train["Age"].fillna(train["Age"].median())
test["Age"] = test["Age"].fillna(train["Age"].median())

# Fill missing Fare in test
test["Fare"] = test["Fare"].fillna(train["Fare"].median())

# Fill missing Embarked
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
test["Embarked"] = test["Embarked"].fillna(train["Embarked"].mode()[0])

# Drop Cabin because it has many missing values
train = train.drop("Cabin", axis=1)
test = test.drop("Cabin", axis=1)


# --------------------------------------------
# 3. FEATURE ENGINEERING
# --------------------------------------------

# Family size
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

# Is the passenger alone?
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)


# --------------------------------------------
# 4. ENCODE CATEGORICAL VARIABLES
# --------------------------------------------

train = pd.get_dummies(
    train,
    columns=["Sex", "Embarked"],
    drop_first=True
)

test = pd.get_dummies(
    test,
    columns=["Sex", "Embarked"],
    drop_first=True
)


# --------------------------------------------
# 5. CREATE X AND y
# --------------------------------------------

X = train.drop(
    ["Survived", "PassengerId", "Name", "Ticket"],
    axis=1
)

y = train["Survived"]

# Test features
X_test = test.drop(
    ["PassengerId", "Name", "Ticket"],
    axis=1
)

# Make sure test columns are exactly the same as training
X_test = X_test.reindex(columns=X.columns, fill_value=0)


# --------------------------------------------
# 6. TRAIN / VALIDATION SPLIT
# --------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# --------------------------------------------
# 7. XGBOOST MODEL
# --------------------------------------------

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)


# --------------------------------------------
# 8. VALIDATION PREDICTION
# --------------------------------------------

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("\nValidation Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_val, y_pred))


# --------------------------------------------
# 9. TRAIN FINAL MODEL ON ALL TRAINING DATA
# --------------------------------------------

model.fit(X, y)


# --------------------------------------------
# 10. PREDICT TEST DATA
# --------------------------------------------

test_predictions = model.predict(X_test)


# --------------------------------------------
# 11. CREATE SUBMISSION CSV
# --------------------------------------------

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv("submission.csv", index=False)


# --------------------------------------------
# 12. CHECK SUBMISSION
# --------------------------------------------

print("\nSubmission created successfully!")
print(submission.head())
print("\nSubmission shape:", submission.shape)

Train shape: (891, 12)
Test shape: (418, 11)

Validation Accuracy: 0.7988826815642458

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.88      0.84       110
           1       0.78      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.80      0.80      0.80       179


Submission created successfully!
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0

Submission shape: (418, 2)
